In [ ]:
using LinearAlgebra

In [ ]:
#quaternion functions
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

H = [zeros(1,3); I];
T = [1  zeros(1,3);
     zeros(3,1) -I];

function L(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I + hat(q[2:4])]
end

function R(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I - hat(q[2:4])]
end

function G(q)
    return L(q)*H
end

function Q(q)
    return H'*L(q)*R(q)'*H
end

In [ ]:
#model parameters
J0 = Diagonal([1.0; 1.25; 1.5])
V = exp(hat(0.1*randn(3)))
J = V'*J0*V

In [ ]:
#dynamics
function dynamics(x,u)
    q = x[1:4]/norm(x[1:4])
    ω = x[5:7]
    ρ = x[8:10]

    ρ̇ = u
    
    q̇ = 0.5*G(q)*ω
    ω̇ = -J\(ρ̇ + hat(ω)*(J*ω + ρ))

    ẋ = [q̇; ω̇; ρ̇]
end

In [ ]:
#Classic RK4 integrator: https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods
function rkstep(x,u)
    f1 = dynamics(x,u)
    f2 = dynamics(x + 0.5*h*f1,u)
    f3 = dynamics(x + 0.5*h*f2,u)
    f4 = dynamics(x + h*f3,u)
    xn = x + (h/6.0)*(f1 + 2*f2 + 2*f3 + f4)
    xn[1:4] .= xn[1:4]/norm(xn[1:4]) #re-normalize quaternion
    return xn
end

In [ ]:
h = 0.1 #time step
n = 600 #number of time steps
tf = n*h #final time

#sample random angular velocity
q0 = [1; 0; 0; 0]
ω0 = [0; 0; 1] + 0.1*randn(3)
#ω0 = V'*[0; 0; 1]

In [ ]:
#Calculate wheel momentum for dynamic balance
ωs = [0; 0; 1];
Js = (ωs/norm(ωs))'*J*(ωs/norm(ωs))
ρs = ωs[3]*(1.2*J[3,3] - Js)
ρ0 = [ωs'; hat(ωs)]\[ρs*ωs[3]; -hat(ωs)*J*ωs]
x0 = [q0; ω0; ρ0];

In [ ]:
#Simulate n time steps
xhist = zeros(10,n)
xhist[:,1] .= x0
u = zeros(3) #no wheel torques
for k = 1:(n-1)
    xhist[:,k+1] = rkstep(xhist[:,k],u)
end

In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations

In [ ]:
vis = Visualizer()
setobject!(vis[:box1], Rect(Vec(-1.5, -1, -0.5), Vec(3, 2, 1)))

In [ ]:
render(vis)

In [ ]:
anim = Animation(vis)

for t = 1:n
    atframe(anim, t) do
        settransform!(vis[:box1], LinearMap(QuatRotation(xhist[1,t], xhist[2,t], xhist[3,t], xhist[4,t])))
    end
end

setanimation!(vis, anim)

In [ ]:
using PythonPlot

In [ ]:
plot(xhist[5,:])
plot(xhist[6,:])
plot(xhist[7,:])

In [ ]:
#Calculate "wobble angle"
atand(norm(ω0[1:2]),ω0[3])